# Cross Validation (교차검증)
모델의 성능을 신뢰할 수 있게 평가하는 방법 중 하나로, 훈련 셋을 여러 개로 나누어 일부를 검증 셋으로 활용하는 방법이다.

## K-Fold
주어진 데이터를 단순히 균등하게 분할하는 방법이다. k는 분할할 fold(세트)의 수를 나타낸다.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# iris 데이터셋
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data,
    iris.target,
    test_size=0.2,
    random_state=42,
    stratify=iris.target
    )

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(120, 4) (120,)
(30, 4) (30,)


## K-Fold
- 가장 기본적인 교차검증 방식으로 데이터를 순서대로 k개의 등분으로 나눈다.
- 데이터가 특정 순서로 정렬 되어 있을 경우 학습이 편향 될 수 있으므로 주의가 필요하다.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

lr_clf = LogisticRegression(max_iter=10000)
kfold = KFold(n_splits=5)
cv_accuracy = []   # 여러 턴의 정확도 관리

# 학습 데이터를 다시 '학습용'과 '검증용' 인덱스로 나누기
for train_index, val_index in kfold.split(X_train):
    # 학습/검증 분리된 인덱스
    # print(train_index)
    # print(val_index)

    # 실제 데이터를 인덱스로 슬라이싱 해서 추출
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

    # 각 Fold에서 정답의 분포가 어떤지 확인 (클래스 불균형 확인용)
    print(f"학습 레이블 분포:\n {pd.Series(y_train_fold).value_counts()}")
    print(f"학습 레이블 분포:\n {pd.Series(y_val_fold).value_counts()}")
    print()

    # 모델 학습 : Fold 내의 학습용 데이터로 예측 수행
    lr_clf.fit(X_train_fold, y_train_fold)

    # 모델 예측 : 학습에 쓰이지 않은 검증용 데이터로 예측 수행
    y_pred = lr_clf.predict(X_val_fold)

    # 평가 : 실제 정답과 예측값을 비교해 정확도 계산
    acc_val = accuracy_score(y_val_fold, y_pred)
    cv_accuracy.append(acc_val)

# 5번의 검증 점수를 평균 내어 모델의 일반적인 성능을 추정
print(f'검증 점수 {np.mean(cv_accuracy)}')

# 교차 검증이 끝난 후 한번도 보지 못한 진짜 테스트 데이터로 최종 성능 확인
acc_score = lr_clf.score(X_test, y_test)
print(f"평가점수 : {acc_score}")

학습 레이블 분포:
 0    33
1    32
2    31
Name: count, dtype: int64
학습 레이블 분포:
 2    9
1    8
0    7
Name: count, dtype: int64

학습 레이블 분포:
 1    33
2    32
0    31
Name: count, dtype: int64
학습 레이블 분포:
 0    9
2    8
1    7
Name: count, dtype: int64

학습 레이블 분포:
 2    33
1    32
0    31
Name: count, dtype: int64
학습 레이블 분포:
 0    9
1    8
2    7
Name: count, dtype: int64

학습 레이블 분포:
 0    33
1    33
2    30
Name: count, dtype: int64
학습 레이블 분포:
 2    10
0     7
1     7
Name: count, dtype: int64

학습 레이블 분포:
 2    34
0    32
1    30
Name: count, dtype: int64
학습 레이블 분포:
 1    10
0     8
2     6
Name: count, dtype: int64

검증 점수 0.9666666666666668
평가점수 : 0.9666666666666667


## StratifiedKFold
- 분류 문제에서 타겟 레이블의 비율을 원본과 동일하게 유지하며 데이터를 나눈다.
- 특정 클래스가 학습이나 검증에서 누락되는 것을 방지한다.

In [7]:
from sklearn.model_selection import StratifiedKFold

lr_clf = LogisticRegression(max_iter=10000)
skfold = StratifiedKFold(n_splits=5)
cv_accuracy = []   # 여러 턴의 정확도 관리

# split 시 y_train 을 반드시 인자로 전달해야 비율을 유지할 수 있다.
for train_index, val_index in kfold.split(X_train, y_train):
    # 학습/검증 분리된 인덱스
    # print(train_index)
    # print(val_index)

    # 실제 데이터를 인덱스로 슬라이싱 해서 추출
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

    # 각 Fold에서 정답의 분포가 어떤지 확인 (클래스 불균형 확인용)
    print(f"학습 레이블 분포:\n {pd.Series(y_train_fold).value_counts()}")
    print(f"학습 레이블 분포:\n {pd.Series(y_val_fold).value_counts()}")
    print()

    # 모델 학습 : Fold 내의 학습용 데이터로 예측 수행
    lr_clf.fit(X_train_fold, y_train_fold)

    # 모델 예측 : 학습에 쓰이지 않은 검증용 데이터로 예측 수행
    y_pred = lr_clf.predict(X_val_fold)

    # 평가 : 실제 정답과 예측값을 비교해 정확도 계산
    acc_val = accuracy_score(y_val_fold, y_pred)
    cv_accuracy.append(acc_val)

# 5번의 검증 점수를 평균 내어 모델의 일반적인 성능을 추정
print(f'검증 점수 {np.mean(cv_accuracy)}')

# 교차 검증이 끝난 후 한번도 보지 못한 진짜 테스트 데이터로 최종 성능 확인
acc_score = lr_clf.score(X_test, y_test)
print(f"평가점수 : {acc_score}")

학습 레이블 분포:
 0    33
1    32
2    31
Name: count, dtype: int64
학습 레이블 분포:
 2    9
1    8
0    7
Name: count, dtype: int64

학습 레이블 분포:
 1    33
2    32
0    31
Name: count, dtype: int64
학습 레이블 분포:
 0    9
2    8
1    7
Name: count, dtype: int64

학습 레이블 분포:
 2    33
1    32
0    31
Name: count, dtype: int64
학습 레이블 분포:
 0    9
1    8
2    7
Name: count, dtype: int64

학습 레이블 분포:
 0    33
1    33
2    30
Name: count, dtype: int64
학습 레이블 분포:
 2    10
0     7
1     7
Name: count, dtype: int64

학습 레이블 분포:
 2    34
0    32
1    30
Name: count, dtype: int64
학습 레이블 분포:
 1    10
0     8
2     6
Name: count, dtype: int64

검증 점수 0.9666666666666668
평가점수 : 0.9666666666666667


## cross_val_score
- 교차 검증의 학습, 예측, 평가 과정을 한 번에 수행해주는 함수
- 설정한 cv 횟수만큼 평가 점수를 리스트 형태로 반환

In [ ]:
from sklearn.model_selection import cross_val_score

lr_clf = LogisticRegression(max_iter=3000)
skfold = StratifiedKFold(n_splits=5)

# 내부적으로 학습, 예측, 평가 과정을 자동으로 수행
# cv 인자에 숫자를 넣으면 기본적으로 분류 모델일 경우 StratifiedKFold
# 리턴 값은 각 Fold별 평가지표 점수가 담긴 리스트
acc_scores = cross_val_score(lr_clf, X_train, y_train, cv=skfold,scoring='accuracy')
print(acc_scores)
print(f'검증 점수: {np.mean(acc_scores)}')

# 모델 학습은 별도로 진행
lr_clf.fit(X_train, y_train)
print(f'평가 점수: {lr_clf.score(X_test, y_test)}')

[0.95833333 0.95833333 0.95833333 0.95833333 1.        ]
검증 점수: 0.9666666666666668


## cross_validate
- cross_val_score 보다 확장 된 기능을 제공
- 평가 점수 뿐만 아니라 학습 시간, 검증 시간, 학습 된 모델 객체 등 상세한 정보를 딕셔너리로 반환

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, f1_score

# 모델 및 분할 전략
lr_clf = LogisticRegression(max_iter=3000)
skfold = StratifiedKFold(n_splits=5)

# 데이터 로드 및 분할
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data,
    iris.target,
    test_size=0.2,
    random_state=42,
    stratify=iris.target
    )

# F1-Score 커스텀 스코어러 생성
# macro : 각 클래스별 f1 점수를 구한 뒤 단순 산술 평균 (클래스별 가중치 동일)
f1_macro = make_scorer(f1_score, average='macro')


# cross_validate 수행
result = cross_validate(
    lr_clf,
    X_train,
    y_train,
    cv=skfold,
    scoring=f1_macro,            # 스코어링 방식 별도 객체로 생성해서 전달
    return_estimator=True        # 학습 된 모델 결과 값에 포함해서 반환
)

# 결과 딕셔너리의 키 값 확인
print("결과 항목 : ", result.keys())

# 결과 분석 및 최적 모델 선택
fit_times = result['fit_time']
score_times = result['score_time']
test_scores = result['test_score']
estimators = result['estimator']

print(f"학습 시간 : {fit_times}")
print(f"평가 시간 : {score_times}")
print(f"전체 검증 점수(f1) : {test_scores}")
print(f"평균 검증 점수(f1) : {np.mean(test_scores)}")

# 점수가 가장 높은 인덱스를 반환
best_score_index = test_scores.argmax()
best_estimator = estimators[best_score_index]

# 최종 평가
# 가장 성적이 좋았던 모델을 꺼내어 테스트 데이터로 최종 성능 확인
y_pred = best_estimator.predict(X_test)
final_f1 = f1_score(y_test, y_pred, average='macro')
print(f"최종 평가 점수(f1): {final_f1}")

결과 항목 :  dict_keys(['fit_time', 'score_time', 'estimator', 'test_score'])
학습 시간 : [0.08368254 0.01193714 0.00945663 0.01043797 0.01253986]
평가 시간 : [0.00151515 0.00244999 0.00200629 0.00200129 0.00301552]
전체 검증 점수(f1) : [0.95816993 0.95816993 0.95816993 0.95816993 1.        ]
평균 검증 점수(f1) : 0.9665359477124185


## 회귀 모델 교차 검증 
- 사이킷런의 scoring 매개변수는 '높을수록 좋은 지표'를 선호
- 따라서 오차 (Error) 지표는 작을수록 좋으므로, 마이너스(-)를 붙여 큰 값이 좋게끔 만들어서 사용한다.

1. R^2(결정계수) : 'scoring='r2''
2. MAE(평균 절대 오차) : 'scoring=neg_mean_absolute_error'로 설정
3. MSE(평균 제곱 오차) : 'socring=neg_mean_squared_error'로 설정
4. RMSE(평균 제곱근 오차) : 'scoring=neg_root_mean_squared_error'로 설정

In [16]:
# 데이터 로드
# 길이, 높이, 너비를 바탕으로 무게를 예측하는 회귀 문제
perch_df = pd.read_csv('data/perch_full.csv')
perch_df.head()

,length,height,width,weight
0,8.4,2.11,1.41,5.9
1,13.7,3.53,2.00,32.0
2,15.0,3.82,2.43,40.0
3,16.2,4.59,2.63,51.5
4,17.4,4.59,2.94,70.0


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 입력/라벨 분리
X = perch_df[['length','height','width']]
y = perch_df['weight']

# 훈련/테스트 분리
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

# 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [20]:
# 회귀 모델 : LinearRegression
from sklearn.linear_model import LinearRegression

# 선형 회귀 모델
regressor = LinearRegression()

# 평가 지표별 교차 검증
r2_scores = cross_val_score(regressor, X_train,y_train, cv=5,scoring='r2')
neg_mse_scores = cross_val_score(regressor, X_train,y_train, cv=5,scoring='neg_mean_squared_error')
neg_rmse_scores = cross_val_score(regressor, X_train,y_train, cv=5,scoring='neg_root_mean_squared_error')
neg_mae_scores = cross_val_score(regressor, X_train,y_train, cv=5,scoring='neg_mean_absolute_error')

In [19]:
# 여러 지표를 한 번에 계산하기
scoring_metrics = {
    'r2' : 'r2',
    'mse' : 'neg_mean_squared_error',
    'mae' : 'neg_mean_absoulute_error'
}

# cross_validate는 여러 개의 scoring을 리스크나 딕셔너리로 전달 할 수 있다
result = cross_validate(regressor, X_train, y_train, cv=5, scoring=scoring_metrics)

print("항목 확인:", result.keys())
print(f"평균 R2: {result['test_r2'].meam()}")
print(f"평균 MSE : {- result['test_mse'].mean()}")
print(f"평균 MAE : {- result['test_mae'].mean()}")

ValueError: 'neg_mean_absoulute_error' is not a valid scoring value. Use sklearn.metrics.get_scorer_names() to get valid options.